# 08. So sánh với scikit-learn, và ảnh hưởng của $\lambda$

Chương 9 và 10 của báo cáo.

Điểm dễ sai nhất: `Ridge` cần $\alpha = \lambda n$ còn `SGDRegressor` cần
$\alpha = \lambda$. Đặt sai không sinh lỗi, chỉ lặng lẽ so hai bài toán khác nhau.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

In [2]:
from src.dataset import load_processed, FULL
from src.reference import run_baselines, baseline_table, save_baselines, verify_alpha_conversion
obj, X_test, y_test, cfg = load_processed("../data/processed", FULL)
verify_alpha_conversion(obj)

{'alpha_ridge': 37947.33192202055,
 'alpha_sgd': 0.03162277660168379,
 'gap': 4.1521364755021424e-30,
 'w_distance': 6.805279175815401e-15,
 'relative_w_distance': 2.3476456227066553e-15,
 'verified': True}

In [3]:
results = run_baselines(obj, X_test, y_test, repeats=3)
save_baselines(results, obj, "../results/raw/full/library.json")
pd.DataFrame(baseline_table(results, obj))

conversion checked: alpha = lam*n = 3.795e+04, |w_sklearn - w*|/|w*| = 2.35e-15, gap = 4.152e-30


   Ridge (solver='auto')          t=  0.271s  gap=  4.152e-30  rmse=3.4646  n_iter=None


   Ridge (solver='cholesky')      t=  0.285s  gap=  4.152e-30  rmse=3.4646  n_iter=None


   Ridge (solver='lsqr')          t=  1.698s  gap=  1.132e-05  rmse=3.4647  n_iter=19


   Ridge (solver='sparse_cg')     t=  2.155s  gap=  1.994e-07  rmse=3.4646  n_iter=None


   Ridge (solver='sag')           t= 43.353s  gap=  3.958e-07  rmse=3.4647  n_iter=42


   SGDRegressor (defaults)        t=  3.033s  gap=  6.212e+20  rmse=33039041537.7940  n_iter=6


   LinearRegression               t=  2.786s  gap=  1.745e-02  rmse=3.4614  n_iter=None


,method,seconds,gap,grad_norm,rmse_test,n_iter
0,closed form (ours),NaN,0.000000e+00,5.719969e-12,NaN,1.0
1,Ridge (solver='auto'),0.270629,4.152136e-30,5.720083e-12,3.464642e+00,NaN
2,Ridge (solver='cholesky'),0.284968,4.152136e-30,5.720083e-12,3.464642e+00,NaN
3,Ridge (solver='lsqr'),1.698281,1.132189e-05,2.183788e-03,3.464660e+00,19.0
4,Ridge (solver='sparse_cg'),2.155463,1.994380e-07,4.070165e-04,3.464642e+00,NaN
5,Ridge (solver='sag'),43.353185,3.957690e-07,1.831774e-04,3.464653e+00,42.0
6,SGDRegressor (defaults),3.032898,6.211851e+20,3.511465e+10,3.303904e+10,6.0
7,LinearRegression,2.785559,1.745040e-02,1.009635e-01,3.461422e+00,NaN


## Ảnh hưởng của hệ số hiệu chỉnh

In [4]:
from src.dataset import SWEEP
from src.experiment import run_lambda_sweep
from src.figures import convergence_pair, lambda_figure, save_figure
sw, Xte, yte, _ = load_processed("../data/processed", SWEEP)
recs, table = run_lambda_sweep(sw.X, sw.y, Xte, yte, out_dir="../results/raw")
pd.DataFrame(table)[["lam", "mu", "kappa", "iters_GD", "iters_AGD", "rmse_test"]]

lambda-kappa: loaded 12 runs from ../results/raw/lambda-kappa.json


,lam,mu,kappa,iters_GD,iters_AGD,rmse_test
0,0.001000,0.003450,2632.961108,NaN,209,3.453746
1,0.010000,0.012450,730.412360,2091.0,118,3.454069
2,0.031623,0.034073,267.527962,766.0,75,3.456312
3,0.100000,0.102450,89.643173,257.0,48,3.469024
4,1.000000,1.002450,10.059336,29.0,17,3.727018
5,10.000000,10.002450,1.907931,5.0,6,4.457777


In [5]:
convergence_pair([r for r in recs if r.meta["method"] == "GD"], "lambda-kappa",
                 title="Effect of lambda on gradient descent", out_dir="../results/figures")
save_figure(lambda_figure(table), "lambda-kappa_tradeoff", "../results/figures")

[PosixPath('../results/figures/lambda-kappa_tradeoff.pdf'),
 PosixPath('../results/figures/lambda-kappa_tradeoff.png')]

Đi từ $\lambda = 0{,}001$ lên $0{,}1$: RMSE xấu đi 0,44%, $\kappa$ giảm 29 lần, và
gradient descent từ chỗ không hội tụ nổi xuống còn 257 vòng.